# Anemo AI — Training Pipeline (Google Colab)

**One-click GPU training for anemia detection using a 10-model CNN ensemble.**

This notebook (15 cells):
1. GPU & environment check
2. Install all dependencies (albumentations v1.x pinned, tensorflowjs, seaborn)
3. Configure Kaggle credentials
4. Download anemia datasets from Kaggle (5 datasets)
5. Organise dataset into 70/15/15 train/val/test splits
6. **Dataset Preview** — class distribution charts, random sample grids, augmentation comparison
7. Train **10 CNN architectures** per body part (conjunctiva, fingernails, skin)
8. **Confusion Matrices** — normalised heatmaps per body part
9. **Grad-CAM Saliency Maps** — visualise model attention on random images per class
10. **Ensemble Weight Config** — AUC-weighted ensemble JSON for app deployment
11. Convert best model to TF.js INT8-quantised format
12. Download all models + configs as ZIP

**Architectures:** EfficientNetV2S · EfficientNetB0 · ResNet50V2 · DenseNet121 · InceptionV3 ·
MobileNetV3Large · Xception · NASNetMobile · EfficientNetB4 · DenseNet201

**Before running:**
- Runtime → Change runtime type → T4 GPU → Save
- Run all cells top-to-bottom (Runtime → Run all, or Shift+Enter per cell)

**Expected time:** ~4–8 h on T4 GPU for all 10 architectures × 3 body parts

In [ ]:
# ── Cell 1: GPU & Environment Check ──────────────────────────────────────────
import subprocess
import sys

try:
    result = subprocess.run(
        ['nvidia-smi'], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print(result.stdout)
    else:
        print('nvidia-smi error (GPU may still be available via TF)')
except (FileNotFoundError, subprocess.TimeoutExpired, OSError):
    print('nvidia-smi not found - normal in some Colab environments')

import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')
for gpu in gpus:
    try:
        details = tf.config.experimental.get_device_details(gpu)
        print(f'  -> {details.get("device_name", str(gpu))}')
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print(f'  -> {gpu} (details unavailable: {e})')

if not gpus:
    print('\nWARNING: No GPU found. Training will be VERY slow on CPU.')
    print('  -> Enable GPU: Runtime -> Change runtime type -> T4 GPU -> Save')
else:
    print('\nGPU ready!')

In [ ]:
# ── Cell 2: Install Dependencies ──────────────────────────────────────────────
import subprocess
import sys

packages = [
    'kaggle>=1.6.0',
    'albumentations>=1.3.0,<2.0.0',   # pin v1.x — stable augmentation API
    'tensorflowjs>=4.10.0',
    'scikit-learn>=1.3.0',
    'opencv-python-headless',
    'tqdm',
    'matplotlib',
    'seaborn',
]

print('Installing dependencies...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    capture_output=True, text=True)
if result.returncode == 0:
    print('All dependencies installed')
else:
    print('Some packages may have issues:')
    print(result.stderr[-1000:] if result.stderr else '(no output)')

# Verify imports
failed = []
for pkg in ['albumentations', 'cv2', 'sklearn']:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  OK {pkg} {ver}')
    except ImportError as e:
        print(f'  FAIL {pkg}: {e}')
        failed.append(pkg)
if failed:
    print(f'\nFailed imports: {failed}. Re-run this cell.')
else:
    print('\nAll imports verified!')

In [ ]:
# ── Cell 3: Kaggle Credentials ────────────────────────────────────────────────
import json
import os
from pathlib import Path

# --- SET YOUR CREDENTIALS -----------------------------------------------------
KAGGLE_USERNAME = 'cyrilcquinoviva'
KAGGLE_KEY      = 'KGAT_4c4efbb044d9da5e17af934f94de5acd'
# ------------------------------------------------------------------------------

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY']      = KAGGLE_KEY

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'
kaggle_json.write_text(json.dumps({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}))
kaggle_json.chmod(0o600)

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY']      = KAGGLE_KEY

import kaggle
try:
    kaggle.api.authenticate()
    print(f'Kaggle authenticated as: {KAGGLE_USERNAME}')
except Exception as e:
    print(f'Kaggle auth note: {type(e).__name__}: {e}')
    print('  Verify credentials at: https://www.kaggle.com/settings -> API')

In [ ]:
# ── Cell 4: Setup Workspace ───────────────────────────────────────────────────
from pathlib import Path

WORKSPACE      = Path('/content/anemo_training')
DATASET_RAW    = WORKSPACE / 'dataset' / 'raw'
DATASET_PROC   = WORKSPACE / 'dataset' / 'processed'
MODELS_OUT     = WORKSPACE / 'models_output'
PUBLIC_MODELS  = WORKSPACE / 'public' / 'models'

for d in [DATASET_RAW, DATASET_PROC, MODELS_OUT, PUBLIC_MODELS]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Workspace : {WORKSPACE}')
print(f'  raw     : {DATASET_RAW}')
print(f'  proc    : {DATASET_PROC}')
print(f'  models  : {MODELS_OUT}')
print(f'  deploy  : {PUBLIC_MODELS}')

In [ ]:
# ── Cell 5: Download Datasets ─────────────────────────────────────────────────
# ┌──────────────────────────────────────────────────────────────────────────┐
# │ IMPORTANT — Accept dataset rules BEFORE running this cell               │
# │ Kaggle returns 403 Forbidden if you haven't agreed to each dataset's    │
# │ rules on the website. While logged in, open each URL and click Download  │
# │ (or I Agree) at least once, then re-run this cell:                      │
# │  1. https://www.kaggle.com/datasets/amandam1/anemia-dataset             │
# │  2. https://www.kaggle.com/datasets/longntt2001/anemia-detection-from-nailbeds │
# │  3. https://www.kaggle.com/datasets/kavindra07/conjunctiva-images-anemia │
# │  4. https://www.kaggle.com/datasets/subhajeetdas/anemia-detection-nailbed│
# │  5. https://www.kaggle.com/datasets/thefearlesscoder/nail-dataset-for-blood-hemoglobin-estimation │
# │  6. https://www.kaggle.com/datasets/omkar-thombre/anemia-detection-dataset│
# └──────────────────────────────────────────────────────────────────────────┘
import kaggle
from pathlib import Path

DATASETS = [
    # (dataset_id,                                              dest_subdir,         priority)
    ('amandam1/anemia-dataset',                               'conjunctiva',         1),
    ('kavindra07/conjunctiva-images-anemia',                  'conjunctiva_alt',     2),
    ('omkar-thombre/anemia-detection-dataset',                'conjunctiva_palm',    2),
    ('longntt2001/anemia-detection-from-nailbeds',            'nailbed',             1),
    ('subhajeetdas/anemia-detection-nailbed',                 'nailbed_alt',         2),
    ('thefearlesscoder/nail-dataset-for-blood-hemoglobin-estimation',
                                                              'nailbed_hgb',         2),
    ('ehababoelnaga/anemia-types-classification',             'clinical',            2),
]

downloaded = {}
failed_403 = []

for dataset_id, dest_name, priority in sorted(DATASETS, key=lambda x: x[2]):
    dest = DATASET_RAW / dest_name
    if dest.exists():
        imgs = list(dest.rglob('*.jpg')) + list(dest.rglob('*.png')) + list(dest.rglob('*.jpeg'))
        if len(imgs) > 10:
            print(f'  skip {dataset_id}: {len(imgs)} images already present')
            downloaded[dest_name] = dest
            continue

    dest.mkdir(parents=True, exist_ok=True)
    print(f'  downloading {dataset_id}...')
    try:
        kaggle.api.dataset_download_files(
            dataset_id, path=str(dest), unzip=True, quiet=False)
        imgs = list(dest.rglob('*.jpg')) + list(dest.rglob('*.png')) + list(dest.rglob('*.jpeg'))
        print(f'    OK: {len(imgs)} images')
        if len(imgs) > 0:
            downloaded[dest_name] = dest
        else:
            print(f'    WARNING: downloaded but no images found (may be CSV-only)')
    except Exception as e:
        err = str(e)
        if '403' in err:
            failed_403.append(dataset_id)
            print(f'    403 FORBIDDEN — must accept rules first!')
            print(f'    → Visit: https://www.kaggle.com/datasets/{dataset_id}')
            print(f'    → Click Download, accept any rules, then re-run this cell.')
        else:
            print(f'    FAIL (priority={priority}): {e}')

print(f'\n{"="*60}')
print(f'Downloaded: {len(downloaded)} dataset(s)')
if failed_403:
    print(f'\n⚠  {len(failed_403)} dataset(s) need rule acceptance on Kaggle.com:')
    for d in failed_403:
        print(f'   → https://www.kaggle.com/datasets/{d}')
    print('\nAfter visiting each URL and accepting rules, re-run this cell.')
if not downloaded and not failed_403:
    raise RuntimeError('No datasets downloaded — check Kaggle credentials in Cell 3.')

In [ ]:
# ── Cell 6: Organise Dataset ──────────────────────────────────────────────────
import random
import shutil
from pathlib import Path

random.seed(42)
CLASS_NAMES = ['0_Normal', '1_Mild', '2_Moderate', '3_Severe']


def organise_binary_dataset(raw_dir, output_base, body_part,
                             positive_kw=None, negative_kw=None):
    positive_kw = positive_kw or ['anemia', 'anemic', 'positive', '1']
    negative_kw = negative_kw or ['normal', 'healthy', 'negative', '0', 'non']

    raw_dir = Path(raw_dir)
    if not raw_dir.exists():
        print(f'  not found: {raw_dir}')
        return 0

    exts = {'.jpg', '.jpeg', '.png', '.bmp'}
    imgs = [f for ext in exts for f in raw_dir.rglob(f'*{ext}')]
    if not imgs:
        print(f'  no images in {raw_dir}')
        return 0

    anemic, normal, unclassified = [], [], []
    for img in imgs:
        check_str = str(img).lower() + ' ' + img.parent.name.lower()
        is_pos = any(k in check_str for k in positive_kw)
        is_neg = any(k in check_str for k in negative_kw)
        if is_pos and not is_neg:
            anemic.append(img)
        elif is_neg and not is_pos:
            normal.append(img)
        else:
            unclassified.append(img)

    normal.extend(unclassified)
    print(f'  {body_part}: {len(anemic)} anemic, {len(normal)} normal')

    for label, files in [('1_Mild', anemic), ('0_Normal', normal)]:
        random.shuffle(files)
        n  = len(files)
        n1 = int(n * 0.70)
        n2 = int(n * 0.15)
        splits = {
            'train': files[:n1],
            'val':   files[n1:n1 + n2],
            'test':  files[n1 + n2:],
        }
        for split_name, split_files in splits.items():
            dest = Path(output_base) / body_part / split_name / label
            dest.mkdir(parents=True, exist_ok=True)
            for src in split_files:
                try:
                    shutil.copy2(src, dest / src.name)
                except Exception:
                    pass
    return len(imgs)


print('Organising datasets...')
total = 0
for conjunctiva_src in ['conjunctiva', 'conjunctiva_palm']:
    total += organise_binary_dataset(DATASET_RAW / conjunctiva_src, DATASET_PROC, 'conjunctiva')
for nail_src in ['nailbed', 'nailbed_hgb']:
    total += organise_binary_dataset(DATASET_RAW / nail_src, DATASET_PROC, 'fingernails')

print(f'\nTotal images organised: {total}')
print('\nDataset summary:')
for bp in ['conjunctiva', 'fingernails', 'skin']:
    bp_dir = DATASET_PROC / bp
    if not bp_dir.exists():
        print(f'  {bp}: no data')
        continue
    n = sum(1 for f in bp_dir.rglob('*.jpg')) + sum(1 for f in bp_dir.rglob('*.png'))
    print(f'  {bp}: {n} images')

In [ ]:
# ── Cell 6b: Dataset Preview & Augmentation Visualization ────────────────────
# Plots class distribution, random sample grids, and augmentation pipeline.
# Self-contained: redefines CLAHE + augmentation pipeline locally.
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import albumentations as A
from pathlib import Path
from PIL import Image

# Local CLAHE for preview (same as training)
def _clahe_preview(img):
    img = np.clip(img, 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge([cl, a, b]), cv2.COLOR_LAB2RGB).astype(np.float32)

# Local augmentation pipeline for preview (v1.x API, safe)
_aug_preview = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.6),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30, p=0.6),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.4),
    A.CoarseDropout(max_holes=8, max_height=24, max_width=24, min_holes=2, p=0.3),
])

CLS_COLORS = {
    '0_Normal': '#27ae60', '1_Mild': '#f39c12',
    '2_Moderate': '#e67e22', '3_Severe': '#c0392b',
}
CLS_LABELS = {
    '0_Normal': 'Normal', '1_Mild': 'Mild Anemia',
    '2_Moderate': 'Moderate Anemia', '3_Severe': 'Severe Anemia',
}
PREVIEW_CLASSES = ['0_Normal', '1_Mild', '2_Moderate', '3_Severe']

def _load_preview_samples(folder, n=4, size=224):
    folder = Path(folder)
    if not folder.exists():
        return []
    paths = list(folder.glob('*.jpg')) + list(folder.glob('*.jpeg')) + list(folder.glob('*.png'))
    random.shuffle(paths)
    imgs = []
    for p in paths:
        if len(imgs) >= n:
            break
        try:
            imgs.append(np.array(Image.open(p).convert('RGB').resize((size, size))))
        except Exception:
            pass
    return imgs

for body_part in ['conjunctiva', 'fingernails', 'skin']:
    train_dir = DATASET_PROC / body_part / 'train'
    if not train_dir.exists():
        print(f'  No processed data for {body_part} — run Cell 6 first')
        continue

    print(f'\n{"=" * 50}')
    print(f'  PREVIEW: {body_part.upper()}')
    print(f'{"=" * 50}')

    # 1. Class Distribution Bar Chart
    counts = {}
    for cn in PREVIEW_CLASSES:
        d = train_dir / cn
        counts[cn] = (
            sum(1 for _ in d.glob('*.jpg')) + sum(1 for _ in d.glob('*.png'))
        ) if d.exists() else 0

    total = sum(counts.values())
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(
        [CLS_LABELS[cn] for cn in PREVIEW_CLASSES],
        [counts[cn] for cn in PREVIEW_CLASSES],
        color=[CLS_COLORS[cn] for cn in PREVIEW_CLASSES],
        edgecolor='white', linewidth=1.5, width=0.6,
    )
    for bar, v in zip(bars, [counts[cn] for cn in PREVIEW_CLASSES]):
        pct = (v / total * 100) if total > 0 else 0
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{v}\n({pct:.1f}%)', ha='center', va='bottom',
                fontweight='bold', fontsize=10)
    ax.set_title(f'{body_part.title()} — Training Set Distribution (total={total})',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of images')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.savefig(str(MODELS_OUT / f'dist_{body_part}.png'), dpi=110, bbox_inches='tight')
    plt.show()
    print(f'  Saved: dist_{body_part}.png')

    # 2. Random Sample Grid (4 classes × 4 images)
    fig, axes = plt.subplots(4, 4, figsize=(14, 14))
    fig.suptitle(f'{body_part.title()} — Random Samples per Class',
                 fontsize=13, fontweight='bold')
    for row, cn in enumerate(PREVIEW_CLASSES):
        samples = _load_preview_samples(train_dir / cn, n=4)
        for col in range(4):
            ax = axes[row][col]
            ax.imshow(samples[col] if col < len(samples)
                      else np.zeros((224, 224, 3), np.uint8))
            ax.axis('off')
            if col == 0:
                ax.set_ylabel(
                    CLS_LABELS[cn], fontsize=10, fontweight='bold',
                    color=CLS_COLORS[cn], rotation=0, labelpad=80, va='center')
    plt.tight_layout()
    plt.savefig(str(MODELS_OUT / f'samples_{body_part}.png'), dpi=110, bbox_inches='tight')
    plt.show()
    print(f'  Saved: samples_{body_part}.png')

    # 3. Augmentation Pipeline Preview
    sample_img = None
    for cn in PREVIEW_CLASSES:
        s = _load_preview_samples(train_dir / cn, n=1)
        if s:
            sample_img = s[0]
            break

    if sample_img is not None:
        n_aug = 5
        fig, axes = plt.subplots(2, n_aug + 1, figsize=(18, 6))
        fig.suptitle(f'{body_part.title()} — Augmentation Pipeline',
                     fontsize=12, fontweight='bold')

        axes[0][0].imshow(sample_img)
        axes[0][0].set_title('Original', fontsize=9)
        axes[0][0].axis('off')
        axes[1][0].imshow(_clahe_preview(sample_img.astype(np.float32)).astype(np.uint8))
        axes[1][0].set_title('+ CLAHE', fontsize=9)
        axes[1][0].axis('off')

        for i in range(1, n_aug + 1):
            aug = _aug_preview(image=sample_img)['image']
            axes[0][i].imshow(aug)
            axes[0][i].set_title(f'Aug {i}', fontsize=9)
            axes[0][i].axis('off')
            axes[1][i].imshow(_clahe_preview(aug.astype(np.float32)).astype(np.uint8))
            axes[1][i].set_title(f'Aug {i} + CLAHE', fontsize=9)
            axes[1][i].axis('off')

        plt.tight_layout()
        plt.savefig(str(MODELS_OUT / f'aug_{body_part}.png'), dpi=110, bbox_inches='tight')
        plt.show()
        print(f'  Saved: aug_{body_part}.png')

print('\nDataset preview complete!')


In [ ]:
# ── Cell 7: Multi-Model Ensemble Training (10 CNN Architectures) ──────────────
#
# Trains up to 10 CNN architectures per body part. Each model is:
#   Phase 1 — head fine-tuned, base frozen          (15 epochs)
#   Phase 2 — top 40% of base unfrozen for fine-tune (30 epochs)
# Models are ranked by AUC on the held-out test set.
# ---
import gc
import json
import numpy as np
import cv2
import albumentations as A
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# ── Mixed precision (GPU speedup) ─────────────────────────────────────────────
try:
    mixed_precision.set_global_policy('mixed_float16')
    print(f'Mixed precision: {mixed_precision.global_policy().name}')
except Exception as e:
    print(f'Mixed precision not available (OK on CPU): {e}')

# ── Configuration ─────────────────────────────────────────────────────────────
LOAD_SIZE   = 224    # base load size (resized per-architecture as needed)
BATCH_SIZE  = 16     # conservative for multi-model memory
NUM_CLASSES = 4
CLASS_NAMES = ['0_Normal', '1_Mild', '2_Moderate', '3_Severe']
SEED        = 42
MAX_MODELS  = 10     # reduce to 3-5 for faster training

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── CLAHE preprocessing ───────────────────────────────────────────────────────
def apply_clahe(img_uint8):
    img = np.clip(img_uint8, 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge([cl, a, b]), cv2.COLOR_LAB2RGB).astype(np.float32)

# ── Augmentation (albumentations v1.x API) ────────────────────────────────────
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.15),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    A.HueSaturationValue(hue_shift_limit=12, sat_shift_limit=25,
                         val_shift_limit=20, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=25,
                       border_mode=cv2.BORDER_REFLECT, p=0.6),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.CoarseDropout(max_holes=8, max_height=24, max_width=24,
                    min_holes=2, p=0.25),
    A.CLAHE(clip_limit=3.0, p=0.3),
    A.ElasticTransform(alpha=30, sigma=5, alpha_affine=5, p=0.1),
])

# ── Architecture Registry (10 CNNs) ───────────────────────────────────────────
def get_model_configs():
    cfgs = {}
    # 1. EfficientNetV2S — best default for medical imaging
    try:
        from tensorflow.keras.applications import EfficientNetV2S
        from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as p
        cfgs['EfficientNetV2S'] = {'builder': EfficientNetV2S, 'preprocess': p, 'size': 224}
    except Exception:
        pass
    # 2. EfficientNetB0 — fast convergence, good baseline
    try:
        from tensorflow.keras.applications import EfficientNetB0
        from tensorflow.keras.applications.efficientnet import preprocess_input as p
        cfgs['EfficientNetB0'] = {'builder': EfficientNetB0, 'preprocess': p, 'size': 224}
    except Exception:
        pass
    # 3. ResNet50V2 — deep residual, strong generalisation
    try:
        from tensorflow.keras.applications import ResNet50V2
        from tensorflow.keras.applications.resnet_v2 import preprocess_input as p
        cfgs['ResNet50V2'] = {'builder': ResNet50V2, 'preprocess': p, 'size': 224}
    except Exception:
        pass
    # 4. DenseNet121 — dense connections, excellent for small medical datasets
    try:
        from tensorflow.keras.applications import DenseNet121
        from tensorflow.keras.applications.densenet import preprocess_input as p
        cfgs['DenseNet121'] = {'builder': DenseNet121, 'preprocess': p, 'size': 224}
    except Exception:
        pass
    # 5. InceptionV3 — multi-scale features
    try:
        from tensorflow.keras.applications import InceptionV3
        from tensorflow.keras.applications.inception_v3 import preprocess_input as p
        cfgs['InceptionV3'] = {'builder': InceptionV3, 'preprocess': p, 'size': 299}
    except Exception:
        pass
    # 6. MobileNetV3Large — efficient, runs well on limited RAM
    try:
        from tensorflow.keras.applications import MobileNetV3Large
        from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as p
        cfgs['MobileNetV3Large'] = {'builder': MobileNetV3Large, 'preprocess': p, 'size': 224}
    except Exception:
        pass
    # 7. Xception — depthwise separable, high accuracy
    try:
        from tensorflow.keras.applications import Xception
        from tensorflow.keras.applications.xception import preprocess_input as p
        cfgs['Xception'] = {'builder': Xception, 'preprocess': p, 'size': 299}
    except Exception:
        pass
    # 8. NASNetMobile — neural architecture search, compact
    try:
        from tensorflow.keras.applications import NASNetMobile
        from tensorflow.keras.applications.nasnet import preprocess_input as p
        cfgs['NASNetMobile'] = {'builder': NASNetMobile, 'preprocess': p, 'size': 224}
    except Exception:
        pass
    # 9. EfficientNetB4 — larger variant, higher capacity
    try:
        from tensorflow.keras.applications import EfficientNetB4
        from tensorflow.keras.applications.efficientnet import preprocess_input as p
        cfgs['EfficientNetB4'] = {'builder': EfficientNetB4, 'preprocess': p, 'size': 380}
    except Exception:
        pass
    # 10. DenseNet201 — deeper dense net for complex patterns
    try:
        from tensorflow.keras.applications import DenseNet201
        from tensorflow.keras.applications.densenet import preprocess_input as p
        cfgs['DenseNet201'] = {'builder': DenseNet201, 'preprocess': p, 'size': 224}
    except Exception:
        pass
    return cfgs

MODELS_CONFIG = dict(list(get_model_configs().items())[:MAX_MODELS])
print(f'Architectures ({len(MODELS_CONFIG)}): {list(MODELS_CONFIG.keys())}')

# ── Ordinal Focal Loss ────────────────────────────────────────────────────────
def ordinal_focal_loss(gamma=2.0, label_smoothing=0.05):
    def loss_fn(y_true, y_pred):
        y_smooth = y_true * (1 - label_smoothing) + label_smoothing / NUM_CLASSES
        ce = tf.keras.losses.categorical_crossentropy(y_smooth, y_pred)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1)
        fl = tf.pow(1.0 - p_t, gamma) * ce
        true_rank = tf.cast(tf.argmax(y_true, axis=-1), tf.float32)
        pred_rank = tf.cast(tf.argmax(y_pred, axis=-1), tf.float32)
        rank_penalty = tf.abs(true_rank - pred_rank) / float(NUM_CLASSES - 1)
        return fl + 0.25 * rank_penalty
    loss_fn.__name__ = 'ordinal_focal_loss'
    return loss_fn

# ── Model Builder ─────────────────────────────────────────────────────────────
def build_arch_model(arch_name, config):
    size = config['size']
    base = config['builder'](weights='imagenet', include_top=False,
                              input_shape=(size, size, 3))
    base.trainable = False
    inp = keras.Input((size, size, 3))
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dense(512, activation='swish')(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(256, activation='swish')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax',
                        dtype='float32', name='predictions')(x)
    return keras.Model(inp, out), base

# ── Data Loader (per-architecture) ────────────────────────────────────────────
def load_split_arch(data_dir, body_part, split, config, augment=False):
    size         = config['size']
    preprocess_fn = config['preprocess']
    images, labels = [], []
    split_dir = Path(data_dir) / body_part / split
    if not split_dir.exists():
        return np.array([]), np.array([])
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = split_dir / class_name
        if not class_dir.exists():
            continue
        img_paths = (list(class_dir.glob('*.jpg')) +
                     list(class_dir.glob('*.jpeg')) +
                     list(class_dir.glob('*.png')))
        for img_path in img_paths:
            try:
                img = img_to_array(
                    load_img(img_path, target_size=(size, size))
                ).astype(np.uint8)
                if augment:
                    img = train_aug(image=img)['image']
                img_proc = preprocess_fn(apply_clahe(img))
                images.append(img_proc)
                labels.append(class_idx)
            except Exception:
                pass
    if not images:
        return np.array([]), np.array([])
    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.int32)
    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

# ── Main Training Loop ────────────────────────────────────────────────────────
trained_models = {}
MODELS_OUT.mkdir(exist_ok=True)
SEP = '=' * 60

for body_part in ['conjunctiva', 'fingernails', 'skin']:
    print(f'\n{SEP}\nTRAINING: {body_part.upper()}\n{SEP}')
    bp_results = {}

    for arch_name, arch_config in MODELS_CONFIG.items():
        arch_size = arch_config['size']
        print(f'\n  [{arch_name}]  input={arch_size}px')

        X_tr, y_tr = load_split_arch(DATASET_PROC, body_part, 'train',
                                      arch_config, augment=True)
        X_va, y_va = load_split_arch(DATASET_PROC, body_part, 'val',
                                      arch_config, augment=False)

        if len(X_tr) == 0:
            print(f'    No data for {body_part} — skipping remaining architectures')
            break
        print(f'    train={len(X_tr)}  val={len(X_va)}')

        y_tr_ohe = np.eye(NUM_CLASSES)[y_tr]
        y_va_ohe = np.eye(NUM_CLASSES)[y_va] if len(y_va) > 0 else None
        val_data = (X_va, y_va_ohe) if y_va_ohe is not None else None

        # Class weights for imbalanced data
        cw_arr = compute_class_weight('balanced',
                                       classes=np.unique(y_tr), y=y_tr)
        class_weights = {int(c): float(w)
                         for c, w in zip(np.unique(y_tr), cw_arr)}
        for i in range(NUM_CLASSES):
            class_weights.setdefault(i, 1.0)

        best_path = str(MODELS_OUT / f'anemia_{body_part}_{arch_name}_best.h5')
        monitor   = 'val_accuracy' if val_data else 'accuracy'
        loss_mon  = 'val_loss'     if val_data else 'loss'

        callbacks = [
            keras.callbacks.ModelCheckpoint(
                best_path, save_best_only=True, monitor=monitor, verbose=0),
            keras.callbacks.EarlyStopping(
                monitor=monitor, patience=7,
                restore_best_weights=True, verbose=1),
            keras.callbacks.ReduceLROnPlateau(
                monitor=loss_mon, factor=0.5, patience=3,
                min_lr=1e-9, verbose=0),
            keras.callbacks.TerminateOnNaN(),
        ]

        try:
            model, base = build_arch_model(arch_name, arch_config)

            # Phase 1: head only (frozen base)
            model.compile(
                optimizer=keras.optimizers.Adam(1e-3),
                loss=ordinal_focal_loss(),
                metrics=['accuracy', keras.metrics.AUC(name='auc')])
            model.fit(X_tr, y_tr_ohe, validation_data=val_data,
                       epochs=15, batch_size=BATCH_SIZE,
                       class_weight=class_weights, callbacks=callbacks,
                       verbose=0)

            # Phase 2: fine-tune top 40% of base
            base.trainable = True
            cut = int(len(base.layers) * 0.6)
            for i, lyr in enumerate(base.layers):
                lyr.trainable = i >= cut
            n_unfrozen = sum(1 for lyr in base.layers if lyr.trainable)
            print(f'    fine-tune {n_unfrozen}/{len(base.layers)} layers')

            model.compile(
                optimizer=keras.optimizers.Adam(2e-6),
                loss=ordinal_focal_loss(),
                metrics=['accuracy', keras.metrics.AUC(name='auc')])
            model.fit(X_tr, y_tr_ohe, validation_data=val_data,
                       epochs=30, batch_size=BATCH_SIZE,
                       class_weight=class_weights, callbacks=callbacks,
                       verbose=0)

            # Load best checkpoint
            if Path(best_path).exists():
                model.load_weights(best_path)

            # Evaluate on test set
            X_te, y_te = load_split_arch(DATASET_PROC, body_part, 'test',
                                          arch_config, augment=False)
            if len(X_te) > 0:
                y_te_ohe = np.eye(NUM_CLASSES)[y_te]
                ev = model.evaluate(X_te, y_te_ohe, verbose=0)
                t_acc, t_auc = ev[1], ev[2]
                y_pred = np.argmax(model.predict(X_te, verbose=0), axis=1)
                print(f'    acc={t_acc:.4f}  AUC={t_auc:.4f}')
                print(classification_report(
                    y_te, y_pred, target_names=CLASS_NAMES, zero_division=0))
            else:
                t_acc = t_auc = 0.0
                print('    trained (no test set)')

            model.save(best_path)
            bp_results[arch_name] = {
                'path':          best_path,
                'test_accuracy': float(t_acc),
                'test_auc':      float(t_auc),
                'input_size':    arch_config['size'],
            }

        except Exception as exc:
            print(f'    FAILED {arch_name}: {exc}')
        finally:
            try:
                del model
            except Exception:
                pass
            tf.keras.backend.clear_session()
            gc.collect()

    if bp_results:
        trained_models[body_part] = bp_results
        ranked = sorted(bp_results.items(),
                        key=lambda x: x[1]['test_auc'], reverse=True)
        print(f'\n  {body_part} ensemble ({len(bp_results)} models):')
        for n, info in ranked:
            print(f'    {n}: acc={info["test_accuracy"]:.4f}  AUC={info["test_auc"]:.4f}')
    else:
        print(f'\n  {body_part}: no models trained (no dataset?)')

print(f'\n{SEP}\nMULTI-MODEL TRAINING COMPLETE\n{SEP}')
for bp, mods in trained_models.items():
    avg_auc = sum(m['test_auc'] for m in mods.values()) / max(len(mods), 1)
    print(f'  {bp}: {len(mods)} models  avg AUC={avg_auc:.4f}')

In [ ]:
# ── Cell 7b: Confusion Matrices per Body Part ────────────────────────────────
# Loads the best model per body part (by AUC) and plots normalised confusion
# matrices side-by-side with raw counts. Requires Cell 7 to have run first.
import gc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report

SHORT_LABELS = ['Normal', 'Mild', 'Moderate', 'Severe']

print('Generating confusion matrices...')

for body_part, models_info in trained_models.items():
    ranked = sorted(models_info.items(), key=lambda x: x[1]['test_auc'], reverse=True)
    arch_name, best_info = ranked[0]
    h5_path = Path(best_info['path'])
    size    = best_info['input_size']

    print(f'\n  {body_part.upper()} — {arch_name} (AUC={best_info["test_auc"]:.4f})')

    if not h5_path.exists():
        print(f'    Model file not found: {h5_path}')
        continue

    try:
        model = tf.keras.models.load_model(str(h5_path), compile=False)
    except Exception as e:
        print(f'    Could not load model: {e}')
        continue

    arch_config = MODELS_CONFIG.get(arch_name)
    if arch_config is None:
        print(f'    Architecture config not found for {arch_name}')
        del model; tf.keras.backend.clear_session(); gc.collect()
        continue

    # Try test split first, fall back to val
    X, y = load_split_arch(DATASET_PROC, body_part, 'test', arch_config, augment=False)
    if len(X) == 0:
        X, y = load_split_arch(DATASET_PROC, body_part, 'val', arch_config, augment=False)
    if len(X) == 0:
        print('    No test/val data available')
        del model; tf.keras.backend.clear_session(); gc.collect()
        continue

    y_pred = np.argmax(model.predict(X, verbose=0, batch_size=16), axis=1)
    del model
    tf.keras.backend.clear_session()
    gc.collect()

    cm_raw  = confusion_matrix(y, y_pred, labels=list(range(NUM_CLASSES)))
    cm_norm = cm_raw.astype('float') / (cm_raw.sum(axis=1, keepdims=True) + 1e-8)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Confusion Matrix — {body_part.title()} ({arch_name})',
                 fontsize=13, fontweight='bold')

    for ax, data, title, fmt in [
        (axes[0], cm_raw,  'Raw Counts',        'd'),
        (axes[1], cm_norm, 'Normalised (row %)', '.2%'),
    ]:
        sns.heatmap(
            data, annot=True, fmt=fmt, cmap='Blues',
            xticklabels=SHORT_LABELS, yticklabels=SHORT_LABELS,
            ax=ax, linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8},
        )
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel('Predicted', fontsize=10)
        ax.set_ylabel('Actual', fontsize=10)

    plt.tight_layout()
    plt.savefig(str(MODELS_OUT / f'confusion_{body_part}.png'), dpi=120, bbox_inches='tight')
    plt.show()
    print(classification_report(y, y_pred, target_names=CLASS_NAMES, zero_division=0))
    print(f'  Saved: confusion_{body_part}.png')

print('\nConfusion matrices complete!')


In [ ]:
# ── Cell 7c: Grad-CAM / Saliency Heatmaps ────────────────────────────────────
# Generates gradient saliency maps for random samples from each class.
# Gradient saliency is architecture-agnostic (works with EfficientNet, etc.)
# and shows pixel-level importance for the predicted class.
import gc
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from PIL import Image

GRADCAM_COLORS = {
    '0_Normal': '#27ae60', '1_Mild': '#f39c12',
    '2_Moderate': '#e67e22', '3_Severe': '#c0392b',
}
GRADCAM_LABELS = {
    '0_Normal': 'Normal', '1_Mild': 'Mild Anemia',
    '2_Moderate': 'Moderate Anemia', '3_Severe': 'Severe Anemia',
}
N_SAMPLES = 3   # images per class per body part

def _compute_saliency(model, img_proc):
    '''Gradient saliency map — works with any Keras architecture.'''
    img_var = tf.Variable(img_proc[np.newaxis].astype('float32'))
    with tf.GradientTape() as tape:
        preds  = model(img_var, training=False)
        top_i  = int(tf.argmax(preds[0]))
        score  = preds[0, top_i]
    grads   = tape.gradient(score, img_var)[0]   # (H, W, 3)
    grads   = tf.abs(grads)
    sal     = tf.reduce_max(grads, axis=-1).numpy()  # (H, W)
    sal_min, sal_max = sal.min(), sal.max()
    sal     = (sal - sal_min) / (sal_max - sal_min + 1e-8)
    return sal, top_i, float(tf.reduce_max(preds[0]))

def _overlay(orig_rgb, saliency):
    '''Overlay colour heatmap on the original image.'''
    h, w    = orig_rgb.shape[:2]
    sal_up  = cv2.resize(saliency, (w, h))
    heat    = cv2.applyColorMap(np.uint8(255 * sal_up), cv2.COLORMAP_JET)
    heat    = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    overlay = 0.55 * orig_rgb.astype(float) + 0.45 * heat.astype(float)
    return np.clip(overlay, 0, 255).astype(np.uint8)

def _load_sample_paths(folder, n):
    folder = Path(folder)
    if not folder.exists():
        return []
    paths = list(folder.glob('*.jpg')) + list(folder.glob('*.jpeg')) + list(folder.glob('*.png'))
    random.shuffle(paths)
    return paths[:n * 3]   # buffer for failures

for body_part, models_info in trained_models.items():
    ranked    = sorted(models_info.items(), key=lambda x: x[1]['test_auc'], reverse=True)
    arch_name, best_info = ranked[0]
    h5_path   = Path(best_info['path'])
    size      = best_info['input_size']

    print(f'\nGrad-CAM: {body_part.upper()} — {arch_name}')
    if not h5_path.exists():
        print(f'  Model not found: {h5_path}')
        continue

    try:
        model = tf.keras.models.load_model(str(h5_path), compile=False)
    except Exception as e:
        print(f'  Could not load model: {e}')
        continue

    arch_config   = MODELS_CONFIG.get(arch_name)
    preprocess_fn = arch_config['preprocess'] if arch_config else (lambda x: x)

    fig_cols = N_SAMPLES * 2
    fig_rows = len(CLASS_NAMES)
    fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(fig_cols * 2.6, fig_rows * 2.8))
    fig.suptitle(
        f'Grad-CAM Saliency Maps: {body_part.title()} ({arch_name})\n'
        'Column pairs: Original | Model Focus Heatmap',
        fontsize=11, fontweight='bold',
    )

    for row, cn in enumerate(CLASS_NAMES):
        test_dir  = DATASET_PROC / body_part / 'test'  / cn
        train_dir = DATASET_PROC / body_part / 'train' / cn
        paths = _load_sample_paths(test_dir, N_SAMPLES)
        if len(paths) < N_SAMPLES:
            paths += _load_sample_paths(train_dir, N_SAMPLES - len(paths))

        drawn = 0
        for img_path in paths:
            if drawn >= N_SAMPLES:
                break
            try:
                orig = np.array(Image.open(img_path).convert('RGB').resize((size, size)))
                proc = preprocess_fn(apply_clahe(orig.astype(np.float32)))
                sal, pred_i, conf = _compute_saliency(model, proc)
                heat_img = _overlay(orig, sal)

                c_o = drawn * 2
                c_h = c_o + 1
                ax_o = axes[row][c_o] if fig_rows > 1 else axes[c_o]
                ax_h = axes[row][c_h] if fig_rows > 1 else axes[c_h]

                ax_o.imshow(orig)
                ax_o.axis('off')
                if drawn == 0:
                    ax_o.set_ylabel(
                        GRADCAM_LABELS.get(cn, cn),
                        fontsize=9, fontweight='bold',
                        color=GRADCAM_COLORS.get(cn, '#888'),
                        rotation=0, labelpad=80, va='center',
                    )

                pred_cn  = CLASS_NAMES[pred_i] if pred_i < len(CLASS_NAMES) else '?'
                correct  = (pred_i == row)
                tc       = '#27ae60' if correct else '#e74c3c'
                verdict  = 'Correct' if correct else 'Wrong'
                pred_lbl = GRADCAM_LABELS.get(pred_cn, pred_cn)
                ax_h.imshow(heat_img)
                ax_h.set_title(
                    f'{verdict} | conf={conf:.2f}\nPred: {pred_lbl}',
                    fontsize=7, color=tc,
                )
                for sp in ax_h.spines.values():
                    sp.set_edgecolor(tc)
                    sp.set_linewidth(2)
                ax_h.axis('off')
                drawn += 1
            except Exception:
                pass

        # Blank unused columns
        for col in range(drawn * 2, fig_cols):
            ax = axes[row][col] if fig_rows > 1 else axes[col]
            ax.axis('off')

    plt.tight_layout()
    save_path = str(MODELS_OUT / f'gradcam_{body_part}.png')
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'  Saved: gradcam_{body_part}.png')

    del model
    tf.keras.backend.clear_session()
    gc.collect()

print('\nGrad-CAM heatmaps complete!')


In [ ]:
# ── Cell 8b: Ensemble Weight Configuration ───────────────────────────────────
# Calculates AUC-weighted ensemble weights for each body part and saves them
# to a JSON config that the app can use at inference time.
import json
from pathlib import Path

print('Computing ensemble weights...')
ensemble_config = {}

for body_part, models_info in trained_models.items():
    total_auc = sum(m['test_auc'] for m in models_info.values())
    weights   = {}
    for arch_name, info in models_info.items():
        w = info['test_auc'] / total_auc if total_auc > 0 else 1.0 / len(models_info)
        weights[arch_name] = {
            'weight':         round(w, 6),
            'test_accuracy':  info['test_accuracy'],
            'test_auc':       info['test_auc'],
            'input_size':     info['input_size'],
            'h5_path':        info['path'],
        }

    ranked = sorted(weights.items(), key=lambda x: x[1]['test_auc'], reverse=True)
    avg_auc = total_auc / max(len(models_info), 1)

    ensemble_config[body_part] = {
        'num_models': len(weights),
        'avg_test_auc': round(avg_auc, 6),
        'best_model': ranked[0][0],
        'models': weights,
    }

    print(f'\n  {body_part.upper()} ({len(weights)} models, avg AUC={avg_auc:.4f}):')
    for name, data in ranked:
        bar = '#' * int(data['weight'] * 50)
        print(f'    {name}: weight={data["weight"]:.4f}  AUC={data["test_auc"]:.4f}  {bar}')

# Save ensemble config
config_path = MODELS_OUT / 'ensemble_config.json'
config_path.write_text(json.dumps(ensemble_config, indent=2))
print(f'\nEnsemble config saved: {config_path}')

# Save a simpler app-ready version
app_config = {}
for bp, cfg in ensemble_config.items():
    app_config[bp] = {
        'bestModel': cfg['best_model'],
        'avgAuc':    cfg['avg_test_auc'],
        'weights': {
            name: data['weight']
            for name, data in cfg['models'].items()
        },
    }
app_config_path = MODELS_OUT / 'ensemble_app_config.json'
app_config_path.write_text(json.dumps(app_config, indent=2))
print(f'App config saved:      {app_config_path}')
print('\nCopy ensemble_app_config.json to src/lib/ensemble/ after training.')


In [ ]:
# ── Cell 8: Convert to TF.js ──────────────────────────────────────────────────
import subprocess
import shutil
from pathlib import Path

# Registry paths must match src/lib/ensemble/model-registry.ts
BODY_PART_REGISTRY = {
    'conjunctiva': [
        'judges/efficientnet-b0',
        'scouts/squeezenet-1.1-eye',
        'specialists/densenet121',
        'judges/vit-tiny',
        'judges/mlp-meta-learner',
    ],
    'fingernails': [
        'scouts/mobilenet-v3-nails',
        'specialists/inceptionv3',
    ],
    'skin': [
        'scouts/mobilenet-v3-skin',
        'specialists/resnet50v2',
        'specialists/vgg16',
    ],
}

converted = []
for body_part, models_info in trained_models.items():
    # Use the highest-AUC model for TF.js conversion
    ranked = sorted(models_info.items(),
                     key=lambda x: x[1]['test_auc'], reverse=True)
    arch_name, best_info = ranked[0]
    h5_path = Path(best_info['path'])

    print(f'\nConverting {body_part} (best: {arch_name}, AUC={best_info["test_auc"]:.4f})...')
    if not h5_path.exists():
        print(f'  Not found: {h5_path}')
        continue

    temp_dir = MODELS_OUT / f'tfjs_{body_part}'
    temp_dir.mkdir(parents=True, exist_ok=True)

    # Try INT8 quantised first, fall back to no quantisation
    for quant_flag in ['--quantize_uint8', None]:
        cmd = [
            'tensorflowjs_converter',
            '--input_format=keras',
            '--output_format=tfjs_graph_model',
        ]
        if quant_flag:
            cmd.append(quant_flag)
        cmd += [str(h5_path), str(temp_dir)]

        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        if result.returncode == 0 and (temp_dir / 'model.json').exists():
            label = 'INT8' if quant_flag else 'none'
            print(f'  Converted (quantisation: {label})')
            break
        else:
            if quant_flag:
                print(f'  INT8 failed, trying without quantisation...')
            else:
                print(f'  Conversion failed: {result.stderr[-200:]}')
                break

    if not (temp_dir / 'model.json').exists():
        continue

    registry_paths = BODY_PART_REGISTRY.get(body_part, ['judges/efficientnet-b0'])
    for reg_path in registry_paths:
        dest = PUBLIC_MODELS / reg_path
        dest.mkdir(parents=True, exist_ok=True)
        for f in temp_dir.iterdir():
            if f.is_file():
                shutil.copy2(f, dest / f.name)
        size_kb = sum(f.stat().st_size for f in dest.rglob('*') if f.is_file()) / 1024
        print(f'  -> public/models/{reg_path} ({size_kb:.0f} KB)')

    shutil.rmtree(temp_dir, ignore_errors=True)
    converted.append(body_part)

print(f'\nTF.js conversion done: {converted}')

In [ ]:
# ── Cell 9: Download Models ───────────────────────────────────────────────────
import shutil
from google.colab import files
from pathlib import Path

print('Creating deployment ZIP...')
export_src = Path('/content/anemo_models_export_tmp')
export_src.mkdir(exist_ok=True)

# Public TF.js models
shutil.copytree(str(PUBLIC_MODELS), str(export_src / 'public' / 'models'),
                dirs_exist_ok=True)

# Keras .h5 files + metadata
h5_dir = export_src / 'keras_models'
h5_dir.mkdir(exist_ok=True)
for h5 in MODELS_OUT.glob('*.h5'):
    shutil.copy2(h5, h5_dir / h5.name)
for meta in MODELS_OUT.glob('*_metadata.json'):
    shutil.copy2(meta, h5_dir / meta.name)

zip_path = '/content/anemo_models_export'
shutil.make_archive(zip_path, 'zip', export_src)
zip_file = zip_path + '.zip'
size_mb = Path(zip_file).stat().st_size / (1024 * 1024)
print(f'ZIP: {zip_file} ({size_mb:.1f} MB)')

print('Downloading...')
files.download(zip_file)
print('\nDownload started!')
print('\nAfter downloading:')
print('  1. Extract ZIP into your project root')
print('  2. Files go to public/models/ (model.json + .bin shards)')
print('  3. Run: npm run dev')
print('  4. Check console for: [ConsensusEngine] Model loaded: ...')

In [ ]:
# ── Cell 10: Final Summary ────────────────────────────────────────────────────
import json
from pathlib import Path

SEP = '=' * 60
print(SEP)
print('ANEMO AI TRAINING SUMMARY')
print(SEP)

print('\nTrained models per body part:')
for bp, models_info in trained_models.items():
    ranked = sorted(models_info.items(),
                     key=lambda x: x[1]['test_auc'], reverse=True)
    avg_auc = sum(m['test_auc'] for m in models_info.values()) / max(len(models_info), 1)
    print(f'\n  {bp.upper()} ({len(models_info)} models, avg AUC={avg_auc:.4f}):')
    for i, (name, info) in enumerate(ranked):
        star = '*' if i == 0 else ' '
        print(f'    {star} {name}: acc={info["test_accuracy"]:.4f}  AUC={info["test_auc"]:.4f}  size={info["input_size"]}px')

print('\nDeployed TF.js models:')
total_kb = 0
for model_json in sorted(PUBLIC_MODELS.rglob('model.json')):
    rel = model_json.parent.relative_to(PUBLIC_MODELS)
    size_kb = sum(f.stat().st_size for f in model_json.parent.rglob('*') if f.is_file()) / 1024
    total_kb += size_kb
    print(f'  /models/{rel}  ({size_kb:.0f} KB)')

print(f'\n  Total: {total_kb:.0f} KB ({total_kb / 1024:.1f} MB)')
print(f'\n{SEP}')
print('All done! Upload the downloaded ZIP to your deployment.')